# V9-B — Micro-batch + bf16: slashing VRAM with zero quality risk

V9-A confirmed the crown (seeded 1.312 vs 1.402, no flips, enwik8/text8 hold,
zero-shot flat to T=2048). V9-B attacks the one remaining tax: peak VRAM.

Key reframing (v9_study_brief §2a): the "fixed-size state" is 8 layers x 256 x
4B = **8 KB**. The 943 MB peak is ~840 MB of **activations held for backward**.
So VRAM reduction needs no architectural change:

1. **Micro-batch** (batch 32 -> 2x16 with grad accumulation): identical
   effective batch -> zero quality risk *by construction*; ~-350-400 MB.
2. **bf16 activations + scan state**: halves activation bytes; params stay
   fp32; the Triton kernel already loads with an explicit cast to fp32, so
   compute stays fp32 in SRAM and only the *stored* h/a go to bf16.

Targets: peak **<= 500 MB** (allocated AND reserved reported), final val
within noise of 1.319, train tok/s >= 58,848, O(T) scaling intact.
The 250-step probe gates bf16 on the full 5-clause trajectory-divergence
acceptance gate (not just final loss); on FAIL the run falls back to
micro-batch-only fp32 and still banks the VRAM win.


## 0. Setup (+ Triton)

In [ ]:
import torch
import triton
print("torch", torch.__version__)
print("triton", triton.__version__)
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
device = "cuda" if torch.cuda.is_available() else "cpu"
assert torch.cuda.is_available(), "V9-B needs the T4 GPU"
assert torch.cuda.is_bf16_supported(), "T4 must support bf16"

def set_seed(s):
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

import math
def check_finite(tag, vals):
    for v in vals:
        if math.isnan(v) or math.isinf(v):
            print(f"NUMERICAL FAULT at {tag}: non-finite value {v} -- STOPPING", flush=True)
            raise SystemExit(f"numerical fault at {tag}")


## 1. Data (campaign corpus, vocab 104)

In [ ]:
import os, urllib.request
os.makedirs("data", exist_ok=True)
URLS = {
    "tinyshakespeare.txt": "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "alice.txt":           "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "frankenstein.txt":    "https://www.gutenberg.org/cache/epub/84/pg84.txt",
    "pride.txt":           "https://www.gutenberg.org/cache/epub/1342/pg1342.txt",
}
parts = []
for name, url in URLS.items():
    p = f"data/{name}"
    if not os.path.exists(p):
        urllib.request.urlretrieve(url, p)
    with open(p, encoding="utf-8", errors="ignore") as f:
        parts.append(f.read())
text = "\n".join(parts)
chars = sorted(set(text)); vocab = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.95 * len(data)); train_data, val_data = data[:n], data[n:]
nl_id = stoi.get("\n")
print(f"corpus {len(text)/1e6:.2f} MB, vocab {vocab}, newline_id={nl_id}")
assert nl_id is not None, "newline not in vocab!"

SEQ_, BS, MB = 128, 32, 16   # effective batch 32 = 2x micro-batch 16
def get_batch(split="train", bs=32):
    d = train_data if split == "train" else val_data
    i = torch.randint(0, len(d) - SEQ_ - 1, (bs,))
    x = torch.stack([d[j:j+SEQ_] for j in i]).to(device)
    y = torch.stack([d[j+1:j+SEQ_+1] for j in i]).to(device)
    return x, y


## 2. Models + Triton kernels (inlined) + V9-B bf16 scan path

In [ ]:
"""
Linear-recurrent replacement for Transformer self-attention.

Experiment: take a standard decoder-only Transformer, rip out the
causal self-attention layer, and replace it with a *moving linear
equation* that compresses the past into a fixed-size running state:

    h_t = d * h_{t-1} + (1 - d) * (B x_t)        (linear recurrence)
    y_t = C h_t                                   (readout)

where d in (0,1)^S is a learned per-channel decay, B/C are learned
projections, and h has FIXED size S independent of sequence length.

This is O(T) time and O(1) memory w.r.t. context length, vs O(T^2)
for attention. The question: how much modeling power do we lose?
"""

import math
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# Token mixing layers
# ----------------------------------------------------------------------------

class SelectiveSegmentedState(nn.Module):
    """V5: gated segmented state-space mixer.

    Replaces the static decay with two dynamic mechanisms:

    1. Dynamic Selective Gate (the model's "will"):
         g_t = sigmoid(W_g x_t + b_g)      in (0,1), per token & channel
       acts as a learned per-token filter: low-information tokens get
       g_t -> 0 (forgotten fast), salient tokens get g_t -> 1 (kept).

    2. Hard Memory Reset Mask (segment boundaries):
         r_t = 1 if token t is a structural boundary (e.g. newline)
         h_t = (1 - r_t) * (g_t * h_{t-1} + (1 - g_t) * u_t)
       when r_t = 1 the state is multiplied by zero: the past memory
       buffer is completely destroyed and the new segment starts fresh.
       (The residual stream still carries x_t itself forward.)

    The recurrence stays affine in h (h_t = a_t*h_{t-1} + b_t with
    a_t = (1-r_t)*g_t, b_t = (1-r_t)*(1-g_t)*u_t), so the parallel
    associative scan still applies: O(T) work, log2(T) parallel passes.
    State dim stays frozen at `state_dim` (VRAM edge preserved).
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)  # bias = per-channel prior
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        # x: (B,T,dim); reset: (B,T) binary, 1 = boundary token
        u = self.in_proj(x)                    # (B,T,S)
        g = torch.sigmoid(self.gate_proj(x))   # (B,T,S) dynamic gate
        if reset is not None:
            keep = (1.0 - reset).unsqueeze(-1)  # (B,T,1)
            a = keep * g
            b = keep * (1.0 - g) * u
        else:
            a = g
            b = (1.0 - g) * u
        h_seq = parallel_scan_ab(a, b)          # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


class CausalSelfAttention(nn.Module):
    """Standard multi-head causal self-attention (the baseline)."""
    def __init__(self, dim, n_heads, dropout=0.0):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)                       # (B,T,H,D)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))  # (B,H,T,D)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(y)


def parallel_scan_ab(A, B):
    """Parallel associative scan (Hillis-Steele) for h_t = A_t*h_{t-1} + B_t.

    A, B: (B, T, S) per-step coefficients. Returns H: (B, T, S).
    log2(T) fully-parallel passes instead of T sequential steps.
    """
    Bsz, T, S = A.shape
    A = A.contiguous()
    Bh = B.contiguous()
    idx = torch.arange(T, device=A.device).view(1, T, 1)
    o = 1
    while o < T:
        m = idx >= o
        A_sh = torch.roll(A, o, dims=1)
        B_sh = torch.roll(Bh, o, dims=1)
        # compose segment [t-o, t-1] then [t]:
        #   a_new = a_t * a_{t-o},  b_new = a_t * b_{t-o} + b_t
        A_new = A * A_sh
        B_new = A * B_sh + Bh
        A = torch.where(m, A_new, A)
        Bh = torch.where(m, B_new, Bh)
        o *= 2
    return Bh


def parallel_linear_recurrence(u, decay):
    """Parallel associative scan (Hillis-Steele) for h_t = d*h_{t-1} + c_t.

    Reformulates the sequential loop as a tree reduction:
      - each step is an affine map  h -> a*h + b, packed as pair (a, b)
      - pair composition is associative:
            (a1,b1) then (a2,b2)  =  (a2*a1, a2*b1 + b2)
      - a parallel prefix over pairs yields every h_t at once

    log2(T) fully-parallel passes instead of T sequential steps.
    u: (B, T, S) inputs already projected; decay: (S,) in (0, 1).
    Returns h: (B, T, S) — identical math to the for-loop version.
    """
    Bsz, T, S = u.shape
    c = (1.0 - decay) * u
    A = decay.expand(Bsz, T, S).contiguous()   # a-coefficients
    return parallel_scan_ab(A, c)


class LinearRunningState(nn.Module):
    """Attention replacement: fixed-size running state, linear recurrence.

    h_t = decay * h_{t-1} + (1 - decay) * (B x_t)
    y_t = C h_t

    - decay: learned per-state-channel forget factor in (0, 1)
    - B:     input  projection  dim -> state_dim
    - C:     output projection  state_dim -> dim
    State size is FIXED (does not grow with context length).
    mode="loop": native Python for-loop (sequential, slow on GPU)
    mode="scan": parallel associative scan (log2(T) parallel passes)
    """
    def __init__(self, dim, state_dim=None, dropout=0.0, mode="loop"):
        super().__init__()
        self.mode = mode
        self.state_dim = state_dim or dim
        # init decay ~ U(0.9, 0.999) -> long-ish memory at start
        init = torch.empty(self.state_dim).uniform_(math.log(0.9 / 0.1),
                                                   math.log(0.999 / 0.001))
        self.logit_decay = nn.Parameter(init)
        self.in_proj = nn.Linear(dim, self.state_dim, bias=False)
        self.out_proj = nn.Linear(self.state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, _ = x.shape
        decay = torch.sigmoid(self.logit_decay)          # (S,) in (0,1)
        u = self.in_proj(x)                              # (B,T,S)
        if self.mode == "scan":
            h_seq = parallel_linear_recurrence(u, decay)  # (B,T,S)
        else:
            keep = 1.0 - decay
            h = torch.zeros(B, self.state_dim, device=x.device, dtype=x.dtype)
            outs = []
            for t in range(T):
                h = decay * h + keep * u[:, t]
                outs.append(h)
            h_seq = torch.stack(outs, dim=1)              # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


# ----------------------------------------------------------------------------
# Blocks & models (identical except for the mixing layer)
# ----------------------------------------------------------------------------

class Block(nn.Module):
    def __init__(self, dim, n_heads, mixer, dropout=0.0, ffn_hidden=None):
        super().__init__()
        ffn_hidden = ffn_hidden or 4 * dim
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.mix = mixer
        self.mlp = nn.Sequential(
            nn.Linear(dim, ffn_hidden), nn.GELU(),
            nn.Linear(ffn_hidden, dim), nn.Dropout(dropout))

    def forward(self, x, reset=None):
        x = x + self.mix(self.ln1(x), reset)
        x = x + self.mlp(self.ln2(x))
        return x


class TinyLM(nn.Module):
    """Decoder-only LM. mixer_fn(dim, n_heads) builds the token-mixing layer."""
    def __init__(self, vocab, dim=128, n_layers=4, n_heads=4,
                 seq_len=128, mixer_fn=None, dropout=0.0, ffn_hidden=None,
                 newline_id=None):
        super().__init__()
        self.seq_len = seq_len
        self.newline_id = newline_id  # boundary token id for reset mask
        self.tok_emb = nn.Embedding(vocab, dim)
        self.pos_emb = nn.Embedding(seq_len, dim)
        self.blocks = nn.ModuleList(
            [Block(dim, n_heads, mixer_fn(dim, n_heads), dropout, ffn_hidden)
             for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(
            torch.arange(T, device=idx.device))[None]
        reset = None
        if self.newline_id is not None:
            reset = (idx == self.newline_id).float()  # (B,T) boundary mask
        for blk in self.blocks:
            x = blk(x, reset)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new, temp=1.0, top_k=None):
        for _ in range(max_new):
            ctx = idx[:, -self.seq_len:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / max(temp, 1e-6)
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def build_models(vocab, dim=128, n_layers=4, n_heads=4, seq_len=128,
                 state_dim=None, dropout=0.0):
    attn = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                  mixer_fn=lambda d, h: CausalSelfAttention(d, h, dropout),
                  dropout=dropout)
    lin = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                 mixer_fn=lambda d, h: LinearRunningState(d, state_dim, dropout),
                 dropout=dropout)
    return attn, lin


"""(inlined) V6/V7/V8-A: Triton-fused scans.

Fuses the V5 recurrence (per the V6 execution plan):
  Step A: the 256-wide hidden state lives in SRAM registers for the whole
          sequence -- loaded once per batch element, never spilled to HBM.
  Step B: sigmoid gate, hard reset mask, and the linear update
          h = keep*g*h + keep*(1-g)*u run in registers, every timestep.
  Step C: only h_t (plus a_t, saved for the backward pass) hit HBM.
          The parallel scan's O(log T) intermediate (B,T,S) tensors vanish.

Math is IDENTICAL to V5's SelectiveSegmentedState -- only the execution
path changes. Zero new parameters.
"""
import triton
import triton.language as tl

BLOCK_S = 256  # state dim; the whole state fits in SRAM


@triton.jit
def _sel_scan_fwd_kernel(u_ptr, gp_ptr, keep_ptr, h_ptr, a_ptr,
                         T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with a_t = keep_t*g_t,
    b_t = keep_t*(1-g_t)*u_t, g_t = sigmoid(gp_t)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # Step A: state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))              # Step B: gate in registers
        at = k * g
        h = at * h + k * (1.0 - g) * u_t             # Step B: update in registers
        tl.store(h_b + t * BLOCK + offs, h)          # Step C: outputs to HBM
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _sel_scan_bwd_kernel(dh_ptr, hs_ptr, ap_ptr, u_ptr, gp_ptr, keep_ptr,
                         du_ptr, dgp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    ap is (B,T+1,S) with ap[t+1] = a_{t+1} (ap[T] = 0).
    D_t = gout_t + a_{t+1}*D_{t+1}; du_t = k(1-g)D_t;
    dgp_t = k*g*(1-g)*(h_{t-1}-u_t)*D_t."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    ap_b = ap_ptr + b * bsp
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dgp_b = dgp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(ap_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))
        omg = 1.0 - g
        tl.store(du_b + t * BLOCK + offs, k * omg * D)
        tl.store(dgp_b + t * BLOCK + offs, k * g * omg * (h_prev - u_t) * D)
        dnext = D


class _SelScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, gp, keep):
        assert u.is_cuda and u.dtype == torch.float32, "V6 kernel: CUDA fp32"
        assert u.shape == gp.shape and u.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, gp, keep = u.contiguous(), gp.contiguous(), keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _sel_scan_fwd_kernel[(B,)](u, gp, keep, h, a, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, gp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, gp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h    # hs[t] = h_{t-1}
        ap = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        ap[:, :T] = a    # ap[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dgp = torch.empty_like(gp)
        _sel_scan_bwd_kernel[(B,)](dh, hs, ap, u, gp, keep, du, dgp, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        return du, dgp, None


def triton_selective_scan(u, gp, keep):
    """Fused selective scan. u, gp: (B,T,256) fp32 CUDA; keep: (B,T) fp32.
    Returns h: (B,T,256). Differentiable."""
    return _SelScanFn.apply(u, gp, keep)


class SelectiveSegmentedStateTriton(nn.Module):
    """V6 mixer: IDENTICAL math and parameters to V5's SelectiveSegmentedState.

    The only change is execution: the recurrence runs in the fused Triton
    kernel (Step A/B/C) instead of the PyTorch Hillis-Steele scan.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V6 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = self.gate_proj(x).float().contiguous()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)
        return self.dropout(self.out_proj(h_seq.to(x.dtype)))


class SelectiveSegmentedStateV7(nn.Module):
    """V7 mixer: V6's Triton-fused selective scan + two literature grafts.

    Graft #1 — output gate + state normalization (HGRN2 / Mamba-2 / GDN):
        y_t = C · (o_t ⊙ RMSNorm(h_t)),   o_t = σ(W_o x_t + b_o)
    States accumulate over segments so readout scale drifts; every strong
    post-Mamba block gates the readout. RMSNorm here is parameter-free
    (no scale vector) to keep the parameter budget exact.

    Graft #3 — layer-graded forget floor (HGRN, NeurIPS'23):
        g_t = σ(W_g x_t + b_g + floor_l)
    floor_l is a per-layer learnable scalar, initialized monotonically
    increasing bottom→top, so low layers forget fast (spelling-level) and
    top layers retain (phrase-level). Complements the newline hard reset.

    The recurrence math is UNCHANGED → the same fused Triton kernel as V6.
    With FFN 1023 the budget lands EXACTLY on the attention baseline.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V7 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)          # (B,T,S)
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


# ----------------------------------------------------------------------------
# V8-A: diagonal gated delta-rule scan
# ----------------------------------------------------------------------------

@triton.jit
def _delta_scan_fwd_kernel(u_ptr, ap_ptr, bp_ptr, kp_ptr, keep_ptr,
                           h_ptr, a_ptr, T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with
        alpha = sigmoid(ap_t), beta = sigmoid(bp_t), kk = sigmoid(kp_t),
        v = silu(u_t),
        a_t = keep_t * alpha * (1 - beta*kk^2),
        b_t = keep_t * beta * kk * v.
    Diagonal restriction of Gated DeltaNet's S_t = a_t S_{t-1}(I-b_t k k^T)
    + b_t v k^T to per-channel 1x1 systems. Strictly contractive:
    sigmoid bounds keep (1-beta*kk^2) in (0,1)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su                                 # SiLU in registers
        at = kr * alpha * (1.0 - beta * kk * kk)
        bt = kr * beta * kk * v
        h = at * h + bt
        tl.store(h_b + t * BLOCK + offs, h)
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _delta_scan_bwd_kernel(dh_ptr, hs_ptr, apn_ptr, u_ptr, ap_ptr, bp_ptr,
                           kp_ptr, keep_ptr, du_ptr, dap_ptr, dbp_ptr,
                           dkp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    apn is (B,T+1,S) with apn[t+1] = a_{t+1} (apn[T] = 0).
    Adjoint: D_t = gout_t + a_{t+1}*D_{t+1}; then per-step
      du  = D*kr*beta*kk*silu'(u)
      dap = D*h_prev*kr*alpha*(1-alpha)*(1-beta*kk^2)
      dbp = D*kr*beta*(1-beta)*(kk*v - alpha*kk^2*h_prev)
      dkp = D*kr*beta*kk*(1-kk)*(v - 2*alpha*kk*h_prev)
    (hand-derived; verified against autograd on CPU before transcription)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    apn_b = apn_ptr + b * bsp
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dap_b = dap_ptr + b * bs
    dbp_b = dbp_ptr + b * bs
    dkp_b = dkp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(apn_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su
        silu_p = su * (1.0 + u_t * (1.0 - su))
        k2 = kk * kk
        omb = 1.0 - beta * k2
        tl.store(du_b + t * BLOCK + offs,
                 D * kr * beta * kk * silu_p)
        tl.store(dap_b + t * BLOCK + offs,
                 D * h_prev * kr * alpha * (1.0 - alpha) * omb)
        tl.store(dbp_b + t * BLOCK + offs,
                 D * kr * beta * (1.0 - beta) * (kk * v - alpha * k2 * h_prev))
        tl.store(dkp_b + t * BLOCK + offs,
                 D * kr * beta * kk * (1.0 - kk) * (v - 2.0 * alpha * kk * h_prev))
        dnext = D


def _delta_scan_cpu(u, ap, bp, kp, keep):
    """CPU reference for the delta scan: differentiable pure-PyTorch loop.

    Used ONLY for local (CPU) verification. On CUDA the fused Triton
    kernel above is used instead (gate-checked against this math on Colab).
    """
    B, T, S = u.shape
    kr = keep.unsqueeze(-1)
    alpha = torch.sigmoid(ap)
    beta = torch.sigmoid(bp)
    kk = torch.sigmoid(kp)
    v = u * torch.sigmoid(u)                          # SiLU
    h = torch.zeros(B, S, dtype=u.dtype, device=u.device)
    hs = []
    for t in range(T):
        a = kr[:, t] * alpha[:, t] * (1.0 - beta[:, t] * kk[:, t] ** 2)
        bb = kr[:, t] * beta[:, t] * kk[:, t] * v[:, t]
        h = a * h + bb
        hs.append(h)
    return torch.stack(hs, dim=1)


class _DeltaScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, ap, bp, kp, keep):
        assert u.is_cuda and u.dtype == torch.float32, \
            "V8-A kernel: CUDA fp32"
        for t in (u, ap, bp, kp):
            assert t.shape == u.shape and t.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, ap, bp, kp = u.contiguous(), ap.contiguous(), bp.contiguous(), kp.contiguous()
        keep = keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _delta_scan_fwd_kernel[(B,)](u, ap, bp, kp, keep, h, a, T,
                                    BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, ap, bp, kp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, ap, bp, kp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h      # hs[t] = h_{t-1}
        apn = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        apn[:, :T] = a     # apn[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dap = torch.empty_like(ap)
        dbp = torch.empty_like(bp)
        dkp = torch.empty_like(kp)
        _delta_scan_bwd_kernel[(B,)](dh, hs, apn, u, ap, bp, kp, keep,
                                    du, dap, dbp, dkp, T,
                                    BLOCK=BLOCK_S, num_warps=4)
        return du, dap, dbp, dkp, None


def triton_delta_scan(u, ap, bp, kp, keep):
    """Fused diagonal delta-rule scan.

    u: raw Bx (SiLU applied in-kernel); ap/bp/kp: pre-sigmoid
    alpha/beta/key projections; keep: (B,T) reset mask.
    (B,T,256) fp32; CUDA -> Triton kernel, CPU -> differentiable
    reference loop (verification only). Returns h: (B,T,256).
    """
    if u.is_cuda:
        return _DeltaScanFn.apply(u, ap, bp, kp, keep)
    return _delta_scan_cpu(u, ap, bp, kp, keep)


class SelectiveSegmentedStateV8A(nn.Module):
    """V8-A mixer: diagonal gated delta rule on V7's chassis.

    Per channel (delta rule = erase-before-write, Gated DeltaNet Eq.10
    restricted to 1x1 systems):
        alpha_t = sigmoid(W_a x_t + b_a + floor_l)   # reuse gate_proj+floor
        beta_t  = sigmoid(W_b x_t + b_b),  b_b = 0   # NEW write_proj
        k_t     = sigmoid(W_k x_t + b_k),  b_k = +2  # NEW key_proj
        v_t     = SiLU(B x_t)                        # reuse in_proj + SiLU
        a_t = alpha_t * (1 - beta_t * k_t^2)
        b_t = beta_t * k_t * v_t
        h_t = (1 - r_t) * (a_t .* h_{t-1} + b_t)
        y_t = C . (o_t .* RMSNorm(h_t)), o_t = sigmoid(W_o x_t)

    Init puts the model near V7's convex-blend regime (k~=0.88, beta~=0.5)
    and the sigmoid bounds keep every step strictly contractive.
    With FFN 766 the budget lands within the relaxed <=3000 gate.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V8-A kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)   # B
        self.gate_proj = nn.Linear(dim, state_dim)             # W_alpha
        self.write_proj = nn.Linear(dim, state_dim)            # W_beta (NEW)
        self.key_proj = nn.Linear(dim, state_dim)              # W_k (NEW)
        self.out_gate_proj = nn.Linear(dim, state_dim)         # W_o
        self.out_proj = nn.Linear(state_dim, dim, bias=False)  # C
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.write_proj.bias.data.zero_()       # b_beta = 0  -> beta ~= 0.5
        self.key_proj.bias.data.fill_(2.0)       # b_k = +2.0  -> k ~= 0.88
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6
        self.last_state = None  # stashed h_seq for stability logging (no grad)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        ap = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        bp = self.write_proj(x).float().contiguous()
        kp = self.key_proj(x).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_delta_scan(u, ap, bp, kp, keep)      # (B,T,S)
        self.last_state = h_seq.detach()  # stability logging only; no numerics change
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


# ---- V9-B: bf16 scan path (appended; triton_kernels.py untouched) ----
class _DeltaScanBF16Fn(torch.autograd.Function):
    # bf16 scan: inputs may be bf16 (kernel loads cast to fp32, so compute
    # is fp32 in SRAM); h/a STORED as bf16 (the VRAM win); backward rebuilds
    # fp32 hs/apn from the bf16 stores (loads cast back). Params stay fp32.
    @staticmethod
    def forward(ctx, u, ap, bp, kp, keep):
        assert u.is_cuda, "V9-B bf16 scan needs CUDA"
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, ap, bp, kp = (t.contiguous() for t in (u, ap, bp, kp))
        keep = keep.contiguous()
        h32 = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a32 = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _delta_scan_fwd_kernel[(B,)](u, ap, bp, kp, keep, h32, a32, T,
                                     BLOCK=BLOCK_S, num_warps=4)
        h16 = h32.to(torch.bfloat16)
        a16 = a32.to(torch.bfloat16)
        ctx.save_for_backward(u, ap, bp, kp, keep, h16, a16)
        return h32

    @staticmethod
    def backward(ctx, dh):
        u, ap, bp, kp, keep, h16, a16 = ctx.saved_tensors
        B, T, S = u.shape
        # fp32 grad buffers: the bwd kernel is then exactly the fp32-gated
        # configuration (no mixed-dtype store semantics to rely on).
        dh = dh.float().contiguous()
        h = h16.float()
        a = a16.float()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h      # hs[t] = h_{t-1}
        apn = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        apn[:, :T] = a     # apn[t+1] = a_{t+1}
        du = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        dap = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        dbp = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        dkp = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _delta_scan_bwd_kernel[(B,)](dh, hs, apn, u.float(), ap.float(),
                                     bp.float(), kp.float(), keep,
                                     du, dap, dbp, dkp, T,
                                     BLOCK=BLOCK_S, num_warps=4)
        return (du.to(u.dtype), dap.to(ap.dtype),
                dbp.to(bp.dtype), dkp.to(kp.dtype), None)


def triton_delta_scan_bf16(u, ap, bp, kp, keep):
    # Fused delta scan with bf16 stores. CUDA -> bf16 path above;
    # CPU -> differentiable fp32 reference (verification only).
    if u.is_cuda:
        return _DeltaScanBF16Fn.apply(u, ap, bp, kp, keep)
    ku, kap, kbp, kkp = (t.float() for t in (u, ap, bp, kp))
    return _delta_scan_cpu(ku, kap, kbp, kkp, keep.float())


class SelectiveSegmentedStateV9B(SelectiveSegmentedStateV8A):
    # V9-B mixer: identical architecture AND parameters to V8-A; only the
    # scan path stores (h, a) as bf16. Run under torch.autocast(bf16):
    # no .float() forcing -- dtypes flow from autocast.
    def forward(self, x, reset=None):
        u = self.in_proj(x).contiguous()
        ap = (self.gate_proj(x) + self.forget_floor).contiguous()
        bp = self.write_proj(x).contiguous()
        kp = self.key_proj(x).contiguous()
        og = self.out_gate_proj(x)
        if reset is not None:
            keep = (1.0 - reset).contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1], device=x.device,
                              dtype=u.dtype)
        h_seq = triton_delta_scan_bf16(u, ap, bp, kp, keep)
        self.last_state = h_seq.detach()  # stability logging only
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


## 3. Param-diff gate — v9b vs attention (diff <= 3000)

In [ ]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())

DIM, LAYERS, HEADS = 256, 8, 8
set_seed(0)
attention = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ_,
    mixer_fn=lambda d, h: CausalSelfAttention(d, h), newline_id=nl_id).to(device)
v9b = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ_,
    mixer_fn=lambda d, h: SelectiveSegmentedStateV9B(d, state_dim=256),
    ffn_hidden=766, newline_id=nl_id).to(device)
_floors = torch.linspace(math.log(0.3/0.7), math.log(0.9/0.1), LAYERS)
for _i, _blk in enumerate(v9b.blocks):
    _blk.mix.forget_floor.data.fill_(_floors[_i])
pa, pv = count_params(attention), count_params(v9b)
print(f"attention={pa} v9b={pv} diff={abs(pa-pv)}")
print("V9-B module params identical to V8-A by construction (same __init__)")
assert abs(pa - pv) <= 3000, "PARAM-DIFF GATE FAILED -- STOPPING RUN"
print("PARAM-DIFF GATE PASSED (diff <= 3000)")
del attention
torch.cuda.empty_cache()


## 4. Micro-batch equivalence (fp32) — 2x16 accumulation == batch 32

In [ ]:
print("=== MICRO-BATCH EQUIVALENCE: 2x16 accumulated grads vs batch-32 grads ===")
set_seed(7)
def _build_v8a(seed):
    set_seed(seed)
    m = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ_,
        mixer_fn=lambda d, h: SelectiveSegmentedStateV8A(d, state_dim=256),
        ffn_hidden=766, newline_id=nl_id).to(device)
    _fl = torch.linspace(math.log(0.3/0.7), math.log(0.9/0.1), LAYERS)
    for _i, _blk in enumerate(m.blocks):
        _blk.mix.forget_floor.data.fill_(_fl[_i])
    return m

m32 = _build_v8a(7)
macc = _build_v8a(0)
macc.load_state_dict(m32.state_dict())   # identical weights
xb32, yb32 = get_batch("train", 32)
# path 1: single batch-32 backward
m32.zero_grad()
_, loss32 = m32(xb32, yb32)
loss32.backward()
g32 = [p.grad.detach().clone() for p in m32.parameters()]
# path 2: 2x16 with loss/2 accumulation
macc.zero_grad()
for k in range(2):
    _, lossk = macc(xb32[k*MB:(k+1)*MB], yb32[k*MB:(k+1)*MB])
    (lossk / 2).backward()
gacc = [p.grad.detach().clone() for p in macc.parameters()]
mdiff = max((a - b).abs().max().item() for a, b in zip(g32, gacc))
print(f"max grad diff (2x16 accum vs batch 32): {mdiff:.2e} (bar 1e-6)")
assert mdiff < 1e-6, "MICRO-BATCH EQUIVALENCE FAILED -- STOPPING RUN"
print("MICRO-BATCH EQUIVALENCE PASSED -- accumulation is exact, zero quality risk by construction")
del m32, macc
torch.cuda.empty_cache()


## 5. bf16 kernel gate — why these bars (read first)

The gate checks **kernel == math**, not fp32 == bf16. The Triton kernel loads
bf16 inputs with an explicit cast to fp32, computes the whole recurrence in
fp32 SRAM registers, and rounds once on store — so the only bf16 effects here
are one input rounding and one output rounding. The manual reference mirrors
exactly that structure (bf16 I/O, fp32 compute, identical op order); residual
differences are fp32 op-ordering noise (~1e-6). The **1e-2 relative bar** is
~5x bf16 eps: loose enough to never false-trip, tight enough to catch any
transcription error in the kernel.

The backward kernel is byte-identical to the fp32-gated one (only the stored
h/a dtypes changed, and loads cast back to fp32) — its check is bf16-path
grads vs fp32-path grads on identical values (pure rounding), bar **5e-2**
for bf16's grad noise floor. The formulas themselves were proven at 1e-5 by
the fp32 gate. bf16's own accumulation epsilon is the *probe's* business,
not the gate's.


In [ ]:
def bf16_kernel_fidelity(B=4, T=128, S=256, verbose=True):
    # Triton kernel (bf16 I/O) vs manual bf16-structured recurrence.
    # Returns (rel_fwd, rel_bwd_max).
    u16 = torch.randn(B, T, S, device=device, dtype=torch.bfloat16)
    ap16 = torch.randn(B, T, S, device=device, dtype=torch.bfloat16)
    bp16 = torch.randn(B, T, S, device=device, dtype=torch.bfloat16)
    kp16 = torch.randn(B, T, S, device=device, dtype=torch.bfloat16)
    keep = (torch.rand(B, T, device=device) > 0.3).float()
    # Triton path: bf16 in -> fp32 compute -> bf16 out
    h32 = torch.empty(B, T, S, device=device, dtype=torch.float32)
    a32 = torch.empty(B, T, S, device=device, dtype=torch.float32)
    _delta_scan_fwd_kernel[(B,)](u16, ap16, bp16, kp16, keep, h32, a32, T,
                                 BLOCK=256, num_warps=4)
    h_tri = h32.to(torch.bfloat16)
    # Reference: same math, bf16 I/O rounding, fp32 compute, same op order
    kr = keep.unsqueeze(-1)
    uf, apf, bpf, kpf = u16.float(), ap16.float(), bp16.float(), kp16.float()
    alpha = torch.sigmoid(apf); beta = torch.sigmoid(bpf); kk = torch.sigmoid(kpf)
    su = torch.sigmoid(uf); v = uf * su
    h = torch.zeros(B, S, device=device); hs = []
    for t in range(T):
        at = kr[:, t] * alpha[:, t] * (1.0 - beta[:, t] * kk[:, t] * kk[:, t])
        bt = kr[:, t] * beta[:, t] * kk[:, t] * v[:, t]
        h = at * h + bt
        hs.append(h.to(torch.bfloat16))
    h_ref = torch.stack(hs, 1)
    denom = h_ref.float().abs().max().clamp(min=1e-6)
    rel_fwd = float((h_tri.float() - h_ref.float()).abs().max() / denom)
    # Backward: bf16-path grads vs fp32-path grads, identical values
    def _grads(use_bf16):
        dt = torch.bfloat16 if use_bf16 else torch.float32
        uu = u16.to(dt).detach().requires_grad_(True)
        aa = ap16.to(dt).detach().requires_grad_(True)
        bb = bp16.to(dt).detach().requires_grad_(True)
        kk_ = kp16.to(dt).detach().requires_grad_(True)
        fn = triton_delta_scan_bf16 if use_bf16 else triton_delta_scan
        fn(uu, aa, bb, kk_, keep).pow(2).sum().backward()
        return [t.grad.float().clone() for t in (uu, aa, bb, kk_)]
    g_b = _grads(True); g_f = _grads(False)
    rel_bwd = max(float((a - b).abs().max() / b.abs().max().clamp(min=1e-6))
                  for a, b in zip(g_b, g_f))
    if verbose:
        print(f"bf16 kernel-vs-math: rel fwd err {rel_fwd:.2e} (bar 1e-2), "
              f"rel bwd err {rel_bwd:.2e} (bar 5e-2)")
    return rel_fwd, rel_bwd

rel_fwd, rel_bwd = bf16_kernel_fidelity()
if not rel_fwd < 1e-2:
    raise SystemExit("BF16 KERNEL GATE FAILED (forward) -- STOPPING RUN")
if not rel_bwd < 5e-2:
    raise SystemExit("BF16 KERNEL GATE FAILED (backward) -- STOPPING RUN")
print("BF16 KERNEL GATE PASSED -- bf16 I/O path through the Triton kernel is faithful")


## 6. 250-step probe — matched seed, fp32 ref vs bf16, both 2x16

In [ ]:
import time
print("=== 250-STEP PROBE (seed 42; fp32 ref vs bf16; identical batches) ===")

def build_pair(seed):
    set_seed(seed)
    ref = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ_,
        mixer_fn=lambda d, h: SelectiveSegmentedStateV8A(d, state_dim=256),
        ffn_hidden=766, newline_id=nl_id).to(device)
    _fl = torch.linspace(math.log(0.3/0.7), math.log(0.9/0.1), LAYERS)
    for _i, _blk in enumerate(ref.blocks):
        _blk.mix.forget_floor.data.fill_(_fl[_i])
    tst = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ_,
        mixer_fn=lambda d, h: SelectiveSegmentedStateV9B(d, state_dim=256),
        ffn_hidden=766, newline_id=nl_id).to(device)
    tst.load_state_dict(ref.state_dict())   # identical init; only precision differs
    return ref, tst

def accum_train_step(m, opt, xb32, yb32, use_amp):
    m.train(); opt.zero_grad()
    tot = 0.0
    for k in range(2):
        xb, yb = xb32[k*MB:(k+1)*MB], yb32[k*MB:(k+1)*MB]
        if use_amp:
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                _, loss = m(xb, yb)
        else:
            _, loss = m(xb, yb)
        (loss / 2).backward()
        tot += float(loss) / 2
    gn = float(torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0))
    check_finite("probe accum step", [tot, gn])
    opt.step()
    return tot, gn

def state_rms_norms(m):
    norms = []
    for blk in m.blocks:
        h = getattr(getattr(blk, 'mix', None), 'last_state', None)
        if h is not None:
            norms.append(float(torch.sqrt((h.float() ** 2).mean()).item()))
    return norms

def timed_accum(m, use_amp, iters=10):
    # fwd+bwd timing only: no opt step, grads cleaned after
    m.train(); m.zero_grad()
    torch.cuda.synchronize(); t0 = time.time()
    for _ in range(iters):
        xb32, yb32 = get_batch("train", 32)
        for k in range(2):
            xb, yb = xb32[k*MB:(k+1)*MB], yb32[k*MB:(k+1)*MB]
            if use_amp:
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    _, loss = m(xb, yb)
            else:
                _, loss = m(xb, yb)
            (loss / 2).backward()
        m.zero_grad()
    torch.cuda.synchronize()
    return iters * 32 * SEQ_ / (time.time() - t0)

def peak_mem(m, use_amp):
    torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
    m.train(); m.zero_grad()
    xb32, yb32 = get_batch("train", 32)
    for k in range(2):
        xb, yb = xb32[k*MB:(k+1)*MB], yb32[k*MB:(k+1)*MB]
        if use_amp:
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                _, loss = m(xb, yb)
        else:
            _, loss = m(xb, yb)
        (loss / 2).backward()
    m.zero_grad()
    alloc = torch.cuda.max_memory_allocated() / 1e6
    reserved = torch.cuda.max_memory_reserved() / 1e6
    torch.cuda.empty_cache()
    return alloc, reserved

VAL_BATCHES = [get_batch("val", 32) for _ in range(10)]
TRN_BATCHES = [get_batch("train", 32) for _ in range(20)]

def probe_measure(m, use_amp, light=False):
    # NO optimizer steps (V4 lesson). Returns the 8 probe metrics.
    was_training = m.training; m.eval()
    with torch.no_grad():
        if use_amp:
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                vl = sum(float(m(xb, yb)[1]) for xb, yb in VAL_BATCHES) / len(VAL_BATCHES)
                tloss_acc = sum(float(m(xb, yb)[1]) for xb, yb in TRN_BATCHES) / len(TRN_BATCHES)
        else:
            vl = sum(float(m(xb, yb)[1]) for xb, yb in VAL_BATCHES) / len(VAL_BATCHES)
            tloss_acc = sum(float(m(xb, yb)[1]) for xb, yb in TRN_BATCHES) / len(TRN_BATCHES)
        sn = float(torch.tensor(state_rms_norms(m)).mean())
    m.train(); m.zero_grad()
    xb0, yb0 = TRN_BATCHES[0]
    for k in range(2):
        xb, yb = xb0[k*MB:(k+1)*MB], yb0[k*MB:(k+1)*MB]
        if use_amp:
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                _, loss = m(xb, yb)
        else:
            _, loss = m(xb, yb)
        (loss / 2).backward()
    gn = float(torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0))
    m.zero_grad()
    out = {"val": vl, "train": tloss_acc, "snorm": sn, "gnorm": gn,
           "kfwd": float("nan"), "kbwd": float("nan"),
           "tps": float("nan"), "mem_alloc": float("nan"), "mem_reserved": float("nan")}
    if not light:
        kfw, kbw = bf16_kernel_fidelity(verbose=False)
        out["kfwd"], out["kbwd"] = kfw, kbw
        out["tps"] = timed_accum(m, use_amp)
        out["mem_alloc"], out["mem_reserved"] = peak_mem(m, use_amp)
    if was_training:
        m.train()
    return out

ref, tst = build_pair(42)
opt_r = torch.optim.AdamW(ref.parameters(), lr=3e-4)
opt_b = torch.optim.AdamW(tst.parameters(), lr=3e-4)
probe = {}
for step in range(1, 251):
    xb32, yb32 = get_batch("train", 32)   # one batch, fed to BOTH models
    accum_train_step(ref, opt_r, xb32, yb32, use_amp=False)
    accum_train_step(tst, opt_b, xb32, yb32, use_amp=True)
    if step in (1, 250):
        mr = probe_measure(ref, False); mb = probe_measure(tst, True)
        probe[step] = (mr, mb)
        print(f"[probe] step {step:3d} dval={mb['val']-mr['val']:+.4f} "
              f"dtrain={mb['train']-mr['train']:+.4f} dsnorm={mb['snorm']-mr['snorm']:+.4f} "
              f"dgnorm={mb['gnorm']-mr['gnorm']:+.4f} kfwd={mb['kfwd']:.2e} kbwd={mb['kbwd']:.2e} "
              f"tps {mr['tps']:.0f}->{mb['tps']:.0f} "
              f"alloc {mr['mem_alloc']:.0f}->{mb['mem_alloc']:.0f}MB "
              f"rsvd {mr['mem_reserved']:.0f}->{mb['mem_reserved']:.0f}MB", flush=True)
print("PROBE TRAINING DONE")


## 7. Probe verdict -> full 1500-step run, or micro-batch-only fallback

In [ ]:
def apply_probe_gate(probe):
    # 5-clause trajectory-divergence acceptance gate (his 13:49 correction):
    # trajectory, not just final loss. Probe evaluates at steps {1, 250};
    # clause (b) uses a probe approximation (b') -- the full 6/7 version
    # runs on the 7-checkpoint full run below.
    trips = []
    steps = sorted(probe)
    dval = {s: probe[s][1]["val"] - probe[s][0]["val"] for s in steps}
    if max(abs(v) for v in dval.values()) > 0.02:                       # (a)
        trips.append("(a) val-loss |bf16-fp32| > 0.02")
    s1, sL = steps[0], steps[-1]                                        # (b')
    if dval[s1] != 0 and (dval[s1] > 0) == (dval[sL] > 0) \
       and abs(dval[sL]) > abs(dval[s1]) and abs(dval[sL]) > 0.005:
        trips.append("(b') consistent growing one-sided drift")
    for s in steps:                                                     # (c)
        mr, mb = probe[s]
        if abs(mb["snorm"] - mr["snorm"]) / max(mr["snorm"], 1e-9) > 0.10:
            trips.append(f"(c) state-norm rel drift >10% @step {s}")
    if max(probe[s][1]["gnorm"] / max(probe[s][0]["gnorm"], 1e-9)      # (d)
           for s in steps) > 1.5:
        trips.append("(d) grad-norm ratio >1.5x")
    for s in steps:                                                     # (e)
        for tag, m_ in (("fp32", probe[s][0]), ("bf16", probe[s][1])):
            for k in ("val", "train", "snorm", "gnorm"):
                v = m_[k]
                if math.isnan(v) or math.isinf(v):
                    trips.append(f"(e) non-finite {k} ({tag}) @step {s}")
    return trips

trips = apply_probe_gate(probe)
PROBE_PASS = not trips
print("PROBE VERDICT:", "PASS -> full 1500-step bf16 run" if PROBE_PASS
      else f"FAIL -- tripped clauses: {trips}")
print("FALLBACK PLAN: micro-batch-only fp32 (still banks the VRAM win)" if not PROBE_PASS else "")

STEPS, EVAL = 1500, 250
CKPTS = [1] + list(range(EVAL, STEPS + 1, EVAL))
full = {}

if PROBE_PASS:
    del ref, tst, opt_r, opt_b
    torch.cuda.empty_cache()
    ref2, tst2 = build_pair(123)
    opt_r2 = torch.optim.AdamW(ref2.parameters(), lr=3e-4)
    opt_b2 = torch.optim.AdamW(tst2.parameters(), lr=3e-4)
    hist = {"fp32": [], "bf16": []}
    gnh = {"fp32": [], "bf16": []}
    snh = {"fp32": [], "bf16": []}
    for step in range(1, STEPS + 1):
        xb32, yb32 = get_batch("train", 32)
        _, gn_r = accum_train_step(ref2, opt_r2, xb32, yb32, use_amp=False)
        _, gn_b = accum_train_step(tst2, opt_b2, xb32, yb32, use_amp=True)
        if step in CKPTS:
            light = step != STEPS
            mr = probe_measure(ref2, False, light=light)
            mb = probe_measure(tst2, True, light=light)
            hist["fp32"].append(mr["val"]); hist["bf16"].append(mb["val"])
            gnh["fp32"].append(mr["gnorm"]); gnh["bf16"].append(mb["gnorm"])
            snh["fp32"].append(mr["snorm"]); snh["bf16"].append(mb["snorm"])
            check_finite(f"full step {step}", [mr["val"], mb["val"], gn_r, gn_b])
            print(f"[full] step {step:5d} fp32 {mr['val']:.3f} | bf16 {mb['val']:.3f} "
                  f"d={mb['val']-mr['val']:+.4f} gnorm {gn_r:.2f}/{gn_b:.2f}", flush=True)
    alloc_b, rsvd_b = mb["mem_alloc"], mb["mem_reserved"]
    tps_b = mb["tps"]
    final_b = hist["bf16"][-1]
    full = {"hist": hist, "gnh": gnh, "snh": snh, "alloc": alloc_b,
            "reserved": rsvd_b, "tps": tps_b, "final": final_b}
    print(f"FULL bf16: final val {final_b:.3f} (bar: within 0.03 of 1.319)")
    print(f"peak alloc {alloc_b:.0f} MB / reserved {rsvd_b:.0f} MB (target: alloc <= 500)")
    print(f"train tok/s {tps_b:.0f} (bar: >= 58848)")
    print("FULL 1500-STEP BF16 RUN DONE")
else:
    print("FALLBACK: micro-batch-only fp32, 1500 steps, seed 123")
    set_seed(123)
    mfb = TinyLM(vocab, DIM, LAYERS, HEADS, SEQ_,
        mixer_fn=lambda d, h: SelectiveSegmentedStateV8A(d, state_dim=256),
        ffn_hidden=766, newline_id=nl_id).to(device)
    _fl = torch.linspace(math.log(0.3/0.7), math.log(0.9/0.1), LAYERS)
    for _i, _blk in enumerate(mfb.blocks):
        _blk.mix.forget_floor.data.fill_(_fl[_i])
    opt_fb = torch.optim.AdamW(mfb.parameters(), lr=3e-4)
    fb_hist = []
    for step in range(1, STEPS + 1):
        xb32, yb32 = get_batch("train", 32)
        _, gn = accum_train_step(mfb, opt_fb, xb32, yb32, use_amp=False)
        if step in CKPTS:
            mr = probe_measure(mfb, False, light=True)
            fb_hist.append(mr["val"])
            check_finite(f"fallback step {step}", [mr["val"], gn])
            print(f"[fb] step {step:5d} val {mr['val']:.3f} gnorm {gn:.2f}", flush=True)
    alloc_f, rsvd_f = peak_mem(mfb, False)
    full = {"fallback_hist": fb_hist, "alloc": alloc_f, "reserved": rsvd_f,
            "final": fb_hist[-1]}
    print(f"FALLBACK DONE: final val {fb_hist[-1]:.3f}, "
          f"peak alloc {alloc_f:.0f} MB / reserved {rsvd_f:.0f} MB")


## 8. O(T) spot-check + full-run trajectory gate (7 checkpoints)

In [ ]:
print("=== O(T) SPOT-CHECK: v9b mixer, bf16, batch 8, fwd+bwd ===")
def bench_bf16(make, T, iters=30):
    m = make().train().to(device)
    x = torch.randn(8, T, 256, device=device)
    r = (torch.rand(8, T, device=device) < 0.3).float()
    for _ in range(5):
        m.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            m(x, r).sum().backward()
    torch.cuda.synchronize(); t0 = time.time()
    for _ in range(iters):
        m.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            m(x, r).sum().backward()
    torch.cuda.synchronize()
    ms = (time.time() - t0) / iters * 1000
    torch.cuda.reset_peak_memory_stats(); m.zero_grad()
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        m(x, r).sum().backward()
    peak = torch.cuda.max_memory_allocated() / 1e6
    torch.cuda.empty_cache()
    return ms, peak

for T in (512, 1024):
    ms, peak = bench_bf16(lambda: SelectiveSegmentedStateV9B(256, 256), T)
    ref_ms = "3.55" if T == 512 else "7.11"
    print(f"  T={T}: {ms:6.2f} ms/fwd+bwd | peak {peak:6.0f} MB  (V8-A fp32: {ref_ms} ms)")
print("SPOT-CHECK DONE")

if PROBE_PASS:
    print("=== FULL-RUN TRAJECTORY GATE (7 checkpoints) ===")
    dvals = [b - a for a, b in zip(full["hist"]["fp32"], full["hist"]["bf16"])]
    trips7 = []
    if max(abs(d) for d in dvals) > 0.02:
        trips7.append("(a) val-loss |bf16-fp32| > 0.02")
    pos = sum(1 for d in dvals if d > 0); neg = sum(1 for d in dvals if d < 0)
    if max(pos, neg) >= 6:
        trips7.append(f"(b) drift: {max(pos,neg)}/7 checkpoints one-sided")
    for i, s in enumerate(CKPTS):
        a, b = full["snh"]["fp32"][i], full["snh"]["bf16"][i]
        if abs(b - a) / max(a, 1e-9) > 0.10:
            trips7.append(f"(c) state-norm drift >10% @step {s}")
    if max(b / max(a, 1e-9) for a, b in
           zip(full["gnh"]["fp32"], full["gnh"]["bf16"])) > 1.5:
        trips7.append("(d) grad-norm ratio >1.5x")
    print("checkpoints:", CKPTS)
    print("dvals:", [f"{d:+.4f}" for d in dvals])
    print("7-CHECKPOINT GATE:", "PASS" if not trips7 else f"TRIPPED {trips7}")
    ok_vram = full["alloc"] <= 500
    ok_val = abs(full["final"] - 1.319) <= 0.03
    ok_tps = full["tps"] >= 58848
    print(f"targets: VRAM<=500MB alloc: {'OK' if ok_vram else 'MISS'} | "
          f"val~1.319: {'OK' if ok_val else 'MISS'} | tok/s>=58848: {'OK' if ok_tps else 'MISS'}")
    print("V9-B VERDICT:", "bf16 ACCEPTED -- all bars hold" if
          (not trips7 and ok_vram and ok_val and ok_tps)
          else "bf16 CONDITIONAL -- see flags above")
else:
    print("V9-B VERDICT: probe failed -- micro-batch-only fallback banked; "
          "bf16 rejected for this run")


## 9. Save everything (results file + Drive best-effort + download)

In [ ]:
lines = ["V9-B micro-batch + bf16 -- run record",
         f"probe: {'PASS' if PROBE_PASS else 'FAIL ' + str(trips)}"]
if PROBE_PASS:
    for i, s in enumerate(CKPTS):
        lines.append(f"ckpt {s}: fp32 {full['hist']['fp32'][i]:.3f} "
                     f"bf16 {full['hist']['bf16'][i]:.3f}")
    lines.append(f"final bf16 {full['final']:.3f} | peak alloc {full['alloc']:.0f} MB "
                 f"| reserved {full['reserved']:.0f} MB | tok/s {full['tps']:.0f}")
else:
    for i, s in enumerate(CKPTS):
        lines.append(f"ckpt {s}: fallback fp32 {full['fallback_hist'][i]:.3f}")
    lines.append(f"fallback final {full['final']:.3f} | peak alloc {full['alloc']:.0f} MB "
                 f"| reserved {full['reserved']:.0f} MB")
open("/content/v9b_results.txt", "w").write("\n".join(lines))
print(open("/content/v9b_results.txt").read())
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    import shutil, datetime
    stamp = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
    shutil.copy("/content/v9b_results.txt",
                f"/content/drive/MyDrive/v9b_results_{stamp}.txt")
    print("Drive copy saved")
except Exception as e:
    print("Drive copy skipped:", e)
try:
    from google.colab import files
    files.download("/content/v9b_results.txt")
    print("download triggered")
except Exception as e:
    print("download skipped:", e)
print("SAVE DONE -- download the .ipynb via File > Download > .ipynb")


## Reading V9-B

- **Micro-batch equivalence** (cell 4) must pass first: it proves the 2x16
  accumulation is bit-exact vs batch 32 *before* bf16 enters the picture.
- **bf16 kernel gate** (cell 5) checks kernel==math at bf16 I/O, not fp32==bf16.
- **250-step probe** (cells 6-7) is the real bf16 test: 8 metrics as
  bf16-fp32 differences, 5-clause trajectory gate. A matching final loss
  with a diverged trajectory FAILS by design.
- On probe FAIL the notebook still delivers value: micro-batch-only fp32
  banks ~-400 MB with zero quality risk.
